# DreemGO · el motor del prototipo, paso a paso

Este notebook usa el mismo código que responde en el API desplegado (`dreemgo/`): carga los artefactos de datos, resuelve consultas reales y mide qué propone el motor en 1 152 consultas. No entrena nada. El motor combina un agrupamiento (los polos), una regla de estacionalidad y una optimización del itinerario; por qué no hay un modelo supervisado está en la [decisión 0004](https://github.com/oswaldoaqm/Alejandros-Team/blob/main/docs/decisiones/0004-sin-modelos-supervisados.md).

**Para correrlo**, desde la raíz del repositorio (o desde la carpeta `codigo/` del zip):

```bash
pip install -e ".[api,dev]"
pip install jupyter
jupyter notebook
```

Tarda unos cinco minutos, casi todo en la rejilla de la sección 4. Las salidas que se ven ya son las de una corrida completa.

## 1 · Los datos que carga el motor

In [1]:
import itertools
import statistics
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path


def raiz_del_codigo() -> Path:
    """La raíz del código: la primera carpeta, de aquí hacia arriba, con pyproject.toml y los artefactos del motor."""
    aqui = Path.cwd().resolve()
    for carpeta in [aqui, *aqui.parents]:
        for candidata in (carpeta, carpeta / "codigo"):
            if (candidata / "pyproject.toml").exists() and (candidata / "dreemgo" / "datos" / "manifiesto.json").exists():
                return candidata
    raise SystemExit("No encuentro el código: abre este notebook dentro del repositorio o junto a la carpeta codigo/.")


RAIZ = raiz_del_codigo()
sys.path.insert(0, str(RAIZ))

from dreemgo.contrato import Consulta  # noqa: E402
from dreemgo.motor.datos import cargar  # noqa: E402
from dreemgo.motor.viaje import resolver  # noqa: E402

datos = cargar()
paradas = sum(1 for r in datos.recursos.values() if r["es_parada"])
print(f"Datos {datos.version}")
print(f"{len(datos.polos)} polos turísticos, {len(datos.recursos)} recursos del inventario dentro de un polo,")
print(f"{paradas} de ellos visitables como parada, {len(datos.eventos)} acontecimientos y {len(datos.origenes)} ciudades de origen.")

Datos 2026.10.4
194 polos turísticos, 6047 recursos del inventario dentro de un polo,
4465 de ellos visitables como parada, 758 acontecimientos y 24 ciudades de origen.


## 2 · Una consulta y su respuesta

El viajero dice desde dónde sale, cuántos días tiene (con la ida y la vuelta), en qué mes viaja y, si quiere, qué le interesa, cuánto piensa gastar y hasta qué altura tolera. El motor devuelve hasta tres viajes, cada uno a un polo distinto, con su plan día por día, su costo en una banda, el veredicto del mes y las razones.

In [2]:
TIPO = {"ida": "ida", "ida_y_visita": "llegada y visitas", "visita": "visitas",
        "visita_y_vuelta": "visitas y vuelta", "vuelta": "vuelta", "ida_visita_y_vuelta": "ida, visitas y vuelta"}


def mostrar(respuesta) -> None:
    """Lo esencial de cada viaje propuesto, como lo ve el viajero en la app."""
    for n, r in enumerate(respuesta.rutas, 1):
        c, e = r.costo, r.estacionalidad
        print(
            f"{n}. {r.polo.nombre} ({r.polo.region}), durmiendo en {r.polo.base.nombre} · "
            f"{r.traslado.horas:.1f} h de ida · S/ {c.p20}-{c.p80} por persona · mes: {e.veredicto}"
        )
        for m in r.motivos[:3]:
            print(f"   · {m}")
        for d in r.dias:
            lugares = ", ".join(p.recurso.nombre for p in d.paradas) or "solo viaje"
            print(f"   día {d.numero} ({TIPO.get(d.tipo, d.tipo)}, {d.horas:.1f} h): {lugares}")
        for a in r.avisos:
            print(f"   ! {a.mensaje}")
        print()


mostrar(resolver(Consulta(origen="cusco", mes=6, dias=2), datos))

1. Urubamba (Cusco), durmiendo en Urubamba · 1.2 h de ida · S/ 385-480 por persona · mes: viable
   · 4 lugares de jerarquía 3 o 4, entre ellos Pueblo de Ollantaytambo, Salineras de Maras y Villa de Urubamba
   · Junio es temporada seca aquí
   · Coincide con Festividad del Señor de Torrechayoc (del 1 de mayo al 30 de junio)
   día 1 (llegada y visitas, 6.8 h): Templo de San Pedro Apóstol de Urubamba., Parque Arqueológico de Ollantaytambo, Pueblo de Ollantaytambo, Mirador de Tantanmarka
   día 2 (visitas y vuelta, 6.6 h): Salineras de Maras, Villa de Urubamba
   ! Se duerme a 2 872 m en Urubamba. Si llegas desde la costa, el cuerpo tarda uno o dos días en acostumbrarse: conviene un primer día tranquilo.

2. Machupicchu Pueblo (Cusco), durmiendo en Machupicchu Pueblo · 3.3 h de ida · S/ 455-682 por persona · mes: viable
   · Parque Arqueológico Nacional de Machu Picchu, de jerarquía 4
   · Junio es temporada seca aquí
   · Coincide con Escenificación de la Rebelión de Manco Inca (21 de 

Desde Lima, cuatro días de julio: Huaraz entra entre las tres, con un día de ida, dos allá y uno de vuelta.

In [3]:
mostrar(resolver(Consulta(origen="lima", mes=7, dias=4), datos))

1. Huaraz (Áncash), durmiendo en Huaraz · 8.1 h de ida · S/ 593-816 por persona · mes: viable
   · 3 lugares de jerarquía 3 o 4: Parque Nacional Huascarán, Quebrada Quilcayhuanca y Museo Arqueológico de Áncash Augusto Soriano Infante
   · Julio es temporada seca aquí
   · Coincide con Festival Ticapampa Pisco Sour Town (del 27 al 28 de julio)
   día 1 (ida, 8.1 h): solo viaje
   día 2 (visitas, 7.8 h): Museo Arqueológico de Áncash Augusto Soriano Infante, Plaza de Armas de Huaraz, Monumento Histórico Puente Calicanto “San Gerónimo” - Huaraz, Parque Nacional Huascarán, Mirador de Rataquenua
   día 3 (visitas, 7.8 h): Complejo Arqueológico Ichic Willcawain, Quebrada Quilcayhuanca, Mirador Natural 24 de Junio, Calle José Olaya - Huaraz, Santuario del Señor de la Soledad - Huaraz, Catedral de Huaraz
   día 4 (vuelta, 8.1 h): solo viaje
   ! Hay paradas a más de 3 500 m: la más alta, Parque Nacional Huascarán, a 4 453 m.
   ! Se duerme a 3 061 m en Huaraz. Si llegas desde la costa, el cuerp

Con intereses y presupuesto: los intereses cambian qué paradas valen más, y el presupuesto reordena sin esconder.

In [4]:
mostrar(resolver(Consulta(origen="arequipa", mes=8, dias=6, intereses=["naturaleza", "aventura"], presupuesto=800), datos))

1. Ilo (Moquegua), durmiendo en Ilo · 4.6 h de ida · S/ 842-1198 por persona · mes: viable
   · 5 lugares de jerarquía 3 o 4, entre ellos Fundo de Olivos Olive Corp (Chiviquina), Fundo de Olivos Poquera y Malecón Costero- Ilo
   · 16 de sus 19 paradas son de naturaleza o aventura
   · Agosto es temporada seca aquí
   día 1 (llegada y visitas, 7.0 h): Valle El Algarrobal
   día 2 (visitas, 5.3 h): Iglesia de San Jerónimo - Ilo, Malecón Costero- Ilo, Fundo de Olivos Olive Corp (Chiviquina), Fundo de Olivos Poquera, Glorieta José Gálvez - Ilo, Muelle Fiscal - Ilo
   día 3 (visitas, 7.7 h): Caleta Puerto Inglés - Ilo, Punta Coles - Rnsiipg, Bodega Lapenta Sac, Fundo Chucarapi
   día 4 (visitas, 7.7 h): Playa del Diablo - Ilo, Fundo La Huerta, Dunas Pampa Salinas - Ilo
   día 5 (visitas, 8.0 h): Bodega Mendoza, Playa Enfermeras-Ilo, Playa Boca del Río - Ilo
   día 6 (visitas y vuelta, 7.8 h): Museo Municipal de Sitio El Algarrobal (Chiribaya), Fundo La Pampita de el Algarrobal - D'Ilo
   ! 

## 3 · La estacionalidad de un polo

Cada mes de cada polo es *viable*, lleva *advertencia* o se *desaconseja*, según diez años de lluvia diaria (Open-Meteo, 2016-2025). Es una regla declarada, no una predicción. Huaraz, por ejemplo:

In [5]:
huaraz = next(p for p in datos.polos.values() if p.base["nombre"] == "Huaraz")
for m in huaraz.clima:
    print(f"mes {m['mes']:>2}: {m['lluvia_mm']:>6.1f} mm de lluvia · {m['veredicto']}")
print(f"Fuente: {huaraz.clima[0]['fuente']}")

mes  1:  222.7 mm de lluvia · desaconsejado
mes  2:  228.4 mm de lluvia · desaconsejado
mes  3:  256.0 mm de lluvia · desaconsejado
mes  4:  156.5 mm de lluvia · advertencia
mes  5:   84.2 mm de lluvia · viable
mes  6:   46.6 mm de lluvia · viable
mes  7:   29.7 mm de lluvia · viable
mes  8:   54.1 mm de lluvia · viable
mes  9:  111.7 mm de lluvia · viable
mes 10:  183.9 mm de lluvia · advertencia
mes 11:  185.6 mm de lluvia · advertencia
mes 12:  218.1 mm de lluvia · advertencia
Fuente: open_meteo_polo


## 4 · Qué propone el motor en 1 152 consultas

Las 24 ciudades de origen, viajes de 2, 4, 6 y 9 días y los doce meses, sin intereses ni presupuesto. Son las cifras de la sección 6 del [informe del prototipo](https://github.com/oswaldoaqm/Alejandros-Team/blob/main/deliveries/week10/PrototypeReport.md). Las rutas son deterministas; los tiempos dependen de la máquina.

In [6]:
rutas, por_consulta, segundos, con_aviso = [], [], [], 0
for origen, dias, mes in itertools.product(sorted(datos.origenes), (2, 4, 6, 9), range(1, 13)):
    inicio = time.perf_counter()
    respuesta = resolver(Consulta(origen=origen, dias=dias, mes=mes), datos)
    segundos.append(time.perf_counter() - inicio)
    por_consulta.append(len(respuesta.rutas))
    for r in respuesta.rutas:
        rutas.append((origen, r.polo.id, r.polo.base.nombre, r.polo.fuera_del_circuito))
        con_aviso += any(a.tipo == "dias" and a.nivel == "advertencia" for a in r.avisos)

polos = Counter(polo for _, polo, _, _ in rutas)
bases = Counter(base for _, _, base, _ in rutas)
diez = sum(n for _, n in polos.most_common(10))
print(f"{len(por_consulta)} consultas: {por_consulta.count(3)} con tres viajes, {len(rutas)} viajes en total")
print(f"Duermen en Machupicchu Pueblo: {bases['Machupicchu Pueblo']} · en Huaraz: {bases['Huaraz']}")
print(f"Fuera del circuito de Lima y Cusco: {sum(f for *_, f in rutas) / len(rutas):.0%}")
print(f"Polos distintos que aparecen: {len(polos)} de {len(datos.polos)} · los diez más propuestos se llevan el {diez / len(rutas):.0%}")
print(f"Con el aviso de que la ida y la vuelta pesan: {con_aviso / len(rutas):.0%}")
print(f"Tiempo por consulta: {statistics.median(segundos):.2f} s de mediana, {max(segundos):.2f} s como máximo")

1152 consultas: 1092 con tres viajes, 3396 viajes en total
Duermen en Machupicchu Pueblo: 9 · en Huaraz: 72
Fuera del circuito de Lima y Cusco: 77%
Polos distintos que aparecen: 63 de 194 · los diez más propuestos se llevan el 57%
Con el aviso de que la ida y la vuelta pesan: 37%
Tiempo por consulta: 0.18 s de mediana, 0.73 s como máximo


## 5 · Lo que el motor garantiza

Diez propiedades del [contrato](https://github.com/oswaldoaqm/Alejandros-Team/blob/main/docs/CONTRATO.md) (§3), que las pruebas verifican sobre consultas generadas al azar. Por ejemplo: ninguna parada supera la altitud pedida, los días suman lo pedido, ninguna jornada con visitas pasa de 8 horas, toda parada enlaza a su ficha oficial y un mes desaconsejado nunca sale sin aviso. Aquí, con 100 consultas; la CI las corre en cada cambio, y con `DREEMGO_PROPIEDADES=1000` son las mil del plan.

In [7]:
corrida = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_propiedades.py", "-p", "no:cacheprovider", "--color=no"],
    cwd=RAIZ, capture_output=True, text=True,
)
print(corrida.stdout.strip().splitlines()[-1])

2 passed in 61.98s (0:01:01)


## 6 · Lo que esto no muestra todavía

- **Machu Picchu solo sale desde el Cusco:** sin vuelos, desde Lima son 24,6 horas por tierra.
- **El valor de una parada (1, 2, 6, 24 según su jerarquía) es un supuesto** que se calibra en la semana 12 con consultas anotadas por el equipo.
- **La distancia del itinerario al óptimo exacto** y una prueba de usabilidad con cinco personas son parte de la evaluación de la semana 12.

Lo demás, con sus cifras, está en el informe del prototipo y en las decisiones del repositorio.